In [3]:
import json
import os
from pprint import pprint

from dotenv import load_dotenv
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
client = genai.Client(api_key=api_key)
print("준비 완료 / 사용 모델:", model)

준비 완료 / 사용 모델: gemini-3.6-flash


In [6]:
# function calling을 통해 외부 api랑 소통해보기.
import requests
def get_exchangerate(country):
    url = f"https://open.er-api.com/v6/latest/{country}"
    response = requests.get(url)

    return response.json()


get_exchangerate_tool = {
    "type": "function",
    "name": "get_exchangerate",
    "description": "국가 통화를 입력받아서 해당 통화의 환율을 반환하는 함수.",
    "parameters": {
        "type": "object",
        "properties": {
            "country": {
                "type": "string",
                "description": "통화. 예시) USD",
            },
        },
        "required": ["country", ],
    },
}


In [8]:
# 위의 과정을 함수로 만들어보자.

TOOL_FUNCTION_MAPPING = {
    "get_exchangerate" : get_exchangerate
}
tools = [get_exchangerate_tool]

def llm_invoke(user_input: str, tools: list[dict], tool_function_mapping):
    max_attemp = 5
    next_input = user_input
    previous_interaction_id = None
    for _ in range(max_attemp):
        # 첫번째 요청
        # 두번째 이상의 요청과의 차이점 : input. previous_interaction_id

        args = {
            "model" : model,
            "input" : next_input,
            # 위에서 작성한 scheme를 전달합니다.
            "tools" : tools,
            # "previous_interaction_id"  :  previous_interaction_id,
            "store" : True,
        }

        # None일 때 아예 args에 들어가지 않도록 합니다.
        if previous_interaction_id is not None:
            args['previous_interaction_id'] = interaction.id

        interaction = client.interactions.create(**args)

        # 이 아래에서
        # next_input을 바꿀꺼야.
        # previous_interaction_id를 바꿀꺼야.
        # if function_call이 없어:
        # model_oupput return해

        function_calls = [step  
                        for step in interaction.steps 
                        if step.type == 'function_call'
                        ]
        
        if not function_calls:
            return interaction.output_text
        
        # function_call이 존재함 -> not function_calls가 False이기 때문에.
        next_input = []
        for function_call in function_calls:
            tool_result = tool_function_mapping[function_call.name](**function_call.arguments)

            print(function_call.name)

            function_result = {
                "type": "function_result",
                "name": function_call.name,
                "call_id": function_call.id,
                "result": [{
                    "type": "text", 
                    "text": json.dumps(tool_result, ensure_ascii=False)}],
            }

            next_input.append(function_result)

        previous_interaction_id = interaction.id


result = llm_invoke("일본과 프랑스의 환율이 어떻게 되는지 알려줘.", tools=tools, tool_function_mapping=TOOL_FUNCTION_MAPPING)

print(result)


get_exchangerate
get_exchangerate
현재 기준 환율 정보입니다. (원화 및 미 달러 기준)

### 🇯🇵 **일본 (엔화, JPY)**
* **100 엔 (JPY)** ≈ **868.72 원 (KRW)**
* **1 엔 (JPY)** ≈ **0.00628 달러 (USD)**

---

### 🇫🇷 **프랑스 (유로, EUR)**
* **1 유로 (EUR)** ≈ **1,614.36 원 (KRW)**
* **1 유로 (EUR)** ≈ **1.167 달러 (USD)**
* **1 유로 (EUR)** ≈ **185.85 엔 (JPY)**
